In [1]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import random
import math
from collections import Counter
from dataclasses import dataclass
from typing import List, Dict

# Détection de police robuste (réutilisée ici)
def get_safe_serif_font():
    available_fonts = {f.name for f in fm.fontManager.ttflist}
    candidates = ['DejaVu Serif', 'Liberation Serif', 'Nimbus Roman', 'serif']
    for font in candidates:
        if font in available_fonts or font == 'serif':
            return font
    return 'serif'

safe_serif = get_safe_serif_font()

plt.rcParams.update({
    'figure.facecolor': '#f4e4bc', 'axes.facecolor': '#f4e4bc',
    'axes.edgecolor': '#2c241b', 'axes.labelcolor': '#2c241b',
    'text.color': '#2c241b', 'xtick.color': '#8b0000', 'ytick.color': '#8b0000',
    'grid.color': '#d4c49c', 'font.family': 'serif', 'font.serif': [safe_serif]
})

# ============================================================
# Fonctions d'activation variées (Le "Chœur Cognitif")
# [FIX] Versions numériquement stables
# ============================================================
def step_activation(x, threshold=0.0):
    return 1.0 if x >= threshold else 0.0

def sigmoid_activation(x, threshold=0.0, steepness=1.0):
    """Sigmoid numériquement stable (Fix overflow)."""
    z = steepness * (x - threshold)
    z = np.clip(z, -500, 500) # Borne l'argument de l'exponentielle
    return 1.0 / (1.0 + np.exp(-z))

def relu_activation(x, threshold=0.0):
    return max(0.0, x - threshold)

def glitch_activation(x, threshold=0.0):
    if abs(x - threshold) < 0.1:
        return -x  # Inversion (Paradoxe)
    return x

ACTIVATIONS = [step_activation, sigmoid_activation, relu_activation, glitch_activation]

class McCullochPittsAttractor:
    def __init__(self, n_neurons=64, connectivity=0.3):
        self.n = n_neurons
        self.W = np.random.randn(n_neurons, n_neurons) * connectivity
        self.W = (self.W + self.W.T) / 2  # Symétrique pour convergence (Hopfield)
        
        # Normalisation spectrale optionnelle pour garantir la stabilité
        eigenvalues = np.linalg.eigvalsh(self.W)
        max_eig = np.max(np.abs(eigenvalues))
        if max_eig > 0.9:
            self.W = self.W * (0.9 / max_eig)
            
        self.thresholds = np.random.uniform(-0.5, 0.5, n_neurons)
        self.activations = np.random.choice(ACTIVATIONS, n_neurons)
        self.state = np.random.choice([0.0, 1.0], n_neurons)

    def update(self):
        new_state = np.zeros(self.n)
        for i in range(self.n):
            input_sum = np.dot(self.W[i], self.state)
            new_state[i] = self.activations[i](input_sum, self.thresholds[i])
        self.state = new_state

    def find_attractor(self, max_steps=1000, tolerance=1e-5):
        history = []
        for _ in range(max_steps):
            self.update()
            history.append(self.state.copy())
            if len(history) > 10:
                diff = np.mean(np.abs(history[-1] - history[-10]))
                if diff < tolerance:
                    break
        return self.state, history

print(" Substrat Neural McCulloch-Pitts chargé.")
print("🎨 Police :", safe_serif)
print(" Fonctions d'activation blindées contre l'overflow.")

 Substrat Neural McCulloch-Pitts chargé.
🎨 Police : DejaVu Serif
 Fonctions d'activation blindées contre l'overflow.


In [2]:
# Notre lexique (simplifié pour l'exemple, mais ce sont les listes du Notebook précédent)
LEXIQUE = {
    "archetypes": ["ombre", "tricheur", "magicien", "soi", "dieu"],
    "biais": ["confirmation", "ancrage", "dissonance", "rareté", "transcendance"],
    "vecteurs_visuels": ["œil_labyrinthe", "miroir_brisé", "foule_clones", "couloir_reflets"],
    "glitchs": ["aberration_chromatique", "bruit_VHS", "calligraphie_quantique"]
}

def decode_genotype_to_phenotype(stable_state):
    """Transforme le vecteur d'attracteur en Mème Textuel et Prompt Visuel."""
    n = len(stable_state)
    
    # 1. Décodage des Archétypes (Premiers 20% des neurones)
    arch_mask = stable_state[:int(n*0.2)] > 0.5
    selected_archs = [LEXIQUE["archetypes"][i % len(LEXIQUE["archetypes"])] 
                      for i, active in enumerate(arch_mask) if active]
    
    # 2. Décodage des Biais (20% à 40%)
    bias_mask = stable_state[int(n*0.2):int(n*0.4)] > 0.5
    selected_biases = [LEXIQUE["biais"][i % len(LEXIQUE["biais"])] 
                       for i, active in enumerate(bias_mask) if active]
                       
    # 3. Décodage Visuel (40% à 80%) - Détermine le vecteur et le glitch
    visual_idx = int(np.sum(stable_state[int(n*0.4):int(n*0.6)]) % len(LEXIQUE["vecteurs_visuels"]))
    glitch_idx = int(np.sum(stable_state[int(n*0.6):int(n*0.8)]) % len(LEXIQUE["glitchs"]))
    
    # 4. Intensité de la charge (80% à 100%)
    charge_intensity = np.mean(stable_state[int(n*0.8):]) * 100
    
    return {
        "archetypes": selected_archs if selected_archs else ["ombre"],
        "biais": selected_biases if selected_biases else ["confirmation"],
        "visuel": LEXIQUE["vecteurs_visuels"][visual_idx],
        "glitch": LEXIQUE["glitchs"][glitch_idx],
        "charge": charge_intensity
    }

In [3]:
def forge_dual_output(phenotype):
    # --- SORTIE 1 : LE MÈME TEXTUEL (Blackmore / Rushkoff) ---
    text_meme = (
        f"Tu penses choisir tes idées, mais ce sont elles qui te choisissent. "
        f"Regarde autour de toi : {phenotype['biais'][0]} n'est pas une erreur de jugement, "
        f"c'est la structure même de ta réalité. "
        f"L'{phenotype['archetypes'][0]} en toi le sait déjà. "
        f"La boucle est bouclée."
    )
    
    # --- SORTIE 2 : LE PROMPT VISUEL (Grok / Gemini) ---
    prompt_image = (
        f"IMAGE CENTRALE : {phenotype['visuel'].replace('_', ' ')}, "
        f"évoquant l'archétype de {phenotype['archetypes'][0]}. "
        f"ARCHITECTURE : Mise en abyme, géométrie non-euclidienne. "
        f"EFFET PSYCHOLOGIQUE : Induire un biais de {phenotype['biais'][0]}, "
        f"charge cognitive à {phenotype['charge']:.0f}%. "
        f"ALTÉRATION : {phenotype['glitch'].replace('_', ' ')}. "
        f"RENDU : Unreal Engine 5, éclairage volumétrique, 8k, photoréaliste, sombre."
    )
    
    return text_meme, prompt_image

In [4]:
# Initialisation du réseau
reseau = McCullochPittsAttractor(n_neurons=64, connectivity=0.4)

# Perturbation initiale (L'exposition au "bruit" médiatique)
print("🌊 Injection du bruit médiatique initial...")
reseau.state = np.random.choice([0.0, 1.0], 64)

# Convergence vers l'attracteur (Le Mème s'installe)
print("🧠 Convergence vers l'état stable (Rétro-Structure)...")
stable_state, history = reseau.find_attractor(max_steps=500)

print(f"✅ Attracteur atteint en {len(history)} itérations.")
print(f"   Énergie finale du réseau : {np.sum(np.abs(stable_state)):.2f}")

# Décodage et Génération
phenotype = decode_genotype_to_phenotype(stable_state)
texte, prompt = forge_dual_output(phenotype)

print("\n" + "="*60)
print(" MÈME TEXTUEL GÉNÉRÉ :")
print("="*60)
print(texte)

print("\n" + "="*60)
print("👁️ PROMPT VISUEL GÉNÉRÉ :")
print("="*60)
print(prompt)

🌊 Injection du bruit médiatique initial...
🧠 Convergence vers l'état stable (Rétro-Structure)...
✅ Attracteur atteint en 500 itérations.
   Énergie finale du réseau : 20.82

 MÈME TEXTUEL GÉNÉRÉ :
Tu penses choisir tes idées, mais ce sont elles qui te choisissent. Regarde autour de toi : ancrage n'est pas une erreur de jugement, c'est la structure même de ta réalité. L'dieu en toi le sait déjà. La boucle est bouclée.

👁️ PROMPT VISUEL GÉNÉRÉ :
IMAGE CENTRALE : couloir reflets, évoquant l'archétype de dieu. ARCHITECTURE : Mise en abyme, géométrie non-euclidienne. EFFET PSYCHOLOGIQUE : Induire un biais de ancrage, charge cognitive à 36%. ALTÉRATION : aberration chromatique. RENDU : Unreal Engine 5, éclairage volumétrique, 8k, photoréaliste, sombre.
